In [6]:
CREATE OR REPLACE VIEW vw_CombinedSales AS
WITH universe AS (
    SELECT
        P.fiscal_year_period_date,
        C.SalesOfficeCode,
        C.CustomerGroupCode,
        P.branch_number,
        CASE WHEN M.PackagingMaterialCode = 'RPCK' THEN 'RPCK' ELSE 'CASE' END AS PackagingMaterialCode,
        C.PrimeVendorDealNumber,
        P.buying_group_code,
        C.CustomerNumber,
        C.RgnPostAcuteSaleGrpNumber,
        P.revenue_amt,
        P.sample_sales_amt,
        P.embrdry_sales_amt,
        P.distributor_rebate_accrual_amt,
        P.group_rebate_accrual_amt,
        P.customer_incentive_rebate_amt,
        P.corporate_rebate_accrual_amt,
        P.invoice_line_number,
        P.sales_order_number
    FROM profitability_analysis_fact P
    JOIN MaterialMaster M ON M.MaterialNumber = P.material_number
    JOIN CustomerMaster C ON C.CustomerNumber = P.customer_number
    WHERE P.fiscal_year_date IN ('2025','2024')
      AND C.SalesOfficeCode <> 'IC'
      AND P.branch_number NOT IN ('DIR','B52','RDM','PDM','B04','LRD','S02','TEM','SAMP','SHR','K44')
      AND P.product_division_code NOT IN ('12','95','83','11','98','13','01')
      AND P.vendor_rebate_accrual_amt = '0.00'
      AND P.billing_type_code = 'F2'
      AND P.material_category_group_code = 'NORM'
      AND P.cost_element_code <> '0000410998'
      AND P.ref_procedure_code NOT IN ('XIPAC','XIPST')
),
segmented_rows AS (
    SELECT
        u.*,
        CASE
            WHEN u.CustomerGroupCode = 'HU' THEN 'CGC_HU'
            WHEN u.CustomerGroupCode = 'UM' THEN 'CGC_UM'
            WHEN u.CustomerGroupCode = 'OP' THEN 'CGC_OP'
            WHEN u.CustomerNumber IN (
                 'CUST_KA1','CUST_KA2','CUST_KA3','CUST_KA4',
                 'CUST_KA5','CUST_KA6','CUST_KA7','CUST_KA8','CUST_KA9'
            ) THEN 'HEALTH_SYSTEM_K'
            WHEN u.buying_group_code IN ('GRP_A','GRP_B')
                 AND u.CustomerNumber IN ('CUSTOMER_A','CUSTOMER_B','CUSTOMER_C',
                                          'CUSTOMER_D','CUSTOMER_E','CUSTOMER_F') THEN 'HEALTH_SYSTEM_A'
            WHEN u.buying_group_code IN ('GRP_C','GRP_D')
                 AND u.RgnPostAcuteSaleGrpNumber = 'SEG_CODE_A' THEN 'HEALTH_SYSTEM_B'
            WHEN u.PrimeVendorDealNumber LIKE '%700646' THEN 'HEALTH_SYSTEM_C'
            WHEN u.buying_group_code = 'GRP_E'  THEN 'HEALTH_SYSTEM_D'
            WHEN u.buying_group_code = 'GRP_F' THEN 'HEALTH_SYSTEM_E'
            ELSE 'General'
        END AS Segment
    FROM universe u
)
SELECT
    fiscal_year_period_date,
    SalesOfficeCode,
    CustomerGroupCode,
    branch_number,
    PackagingMaterialCode,
    Segment,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt) AS GrossSales,
    SUM(revenue_amt + sample_sales_amt + embrdry_sales_amt
        + distributor_rebate_accrual_amt + group_rebate_accrual_amt
        + customer_incentive_rebate_amt + corporate_rebate_accrual_amt) AS NetSales,
    COUNT(invoice_line_number) AS Lines
FROM segmented_rows
GROUP BY
    fiscal_year_period_date, SalesOfficeCode, CustomerGroupCode,
    branch_number, PackagingMaterialCode, Segment;